# Ensemble of two five-fold members and two fold-0 members

Notebook 18 averaged four fold-0 models: Exp1 (ConvNeXt-base), Exp3 (EfficientNetV2-S), Exp5 (BiT ResNetV2-50x1) and Exp7 (HRNet-W32). It scored 0.97816 on the public leaderboard. Exp3 and Exp5 now have five-fold versions (notebooks `cv/03` and `cv/04`): their test probabilities are the mean of five fold models, each trained on a different 80% of the data. This notebook replaces only those two members and keeps the method of notebook 18: equal weights and the fixed threshold 0.25. So any change in the score comes from the five-fold members, not from weights or thresholds.

The notebook only reads saved probabilities. No model is run. The submission CSV and its record are written only with `BUILD_SUBMISSION=True`, and never over an existing file. Kaggle upload is always manual.

In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np

BUILD_SUBMISSION = True
SUBMIT_TO_KAGGLE = False  # No submission API call exists in this notebook.

RUN_ID = 'cv_ensemble_exp1_cvexp3_cvexp5_exp7_mean_025'
THRESHOLD = 0.25
VALIDATION_FOLD = 0
# Member -> its fold-0 run (old members) or its five-fold folder under experiments/cv/ (new members).
FOLD0_MEMBERS = {
    'exp1': ('exp1', 'convnext_standard_fold0_seed42_20261018_full1'),
    'exp07': ('exp07', 'hrnet_w32_finetune_fold0_seed42'),
}
CV_MEMBERS = {'cv_exp3': 'efficientnetv2s', 'cv_exp5': 'bit_r50x1'}
# Earlier submissions to compare with, cell by cell.
EARLIER = {
    'old 4-mean (public 0.97816)': 'submissions/exp4_ensemble_exp1_exp3_exp5_exp7_mean_025.csv',
    'best, skill weights (public 0.98305)': 'submissions/exp4_weighted_skill_exp1_exp3_exp5_exp7_025.csv',
}

start = Path(os.environ.get('STAGE2_ROOT', Path.cwd())).resolve()
ROOT = next((path for path in (start, *start.parents) if (path / 'folds.csv').is_file() and (path / 'stage2' / 'cv.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Launch Jupyter from this repository or a subdirectory, or set STAGE2_ROOT.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## Load the members

The old members give their fold-0 validation probabilities and their saved test probabilities (the same files as in notebook 18). The five-fold members give their validation probabilities for all training images, the fold of each image, and their fold-mean test probabilities; `read_cv_summary` checks each file against the hash in `cv.json`. All four must use the same fold file and label order. Every file is aligned to the image IDs of `data/train.csv` and `data/test.csv`, so the rows match across members.

In [2]:
from stage2.cv import read_cv_summary, sha256
from stage2.ensemble import mean_probabilities, read_probabilities
from stage2.experiment import LABEL_COLUMNS, form_submission, read_csv, score_probabilities, write_run_record, write_submission

DATA_DIR = ROOT / 'data'
train_rows = read_csv(DATA_DIR / 'train.csv', ('image_id', *LABEL_COLUMNS))
train_ids = [row['image_id'] for row in train_rows]
train_targets = np.asarray([[int(row[label]) for label in LABEL_COLUMNS] for row in train_rows])
test_ids = [row['image_id'] for row in read_csv(DATA_DIR / 'test.csv', ('image_id',))]
fold_sha256 = sha256(ROOT / 'folds.csv')

cv_members = {name: read_cv_summary(ROOT / 'experiments' / 'cv' / folder, train_ids, test_ids) for name, folder in CV_MEMBERS.items()}
folds = cv_members['cv_exp3']['folds']
for name, member in cv_members.items():
    data = member['record']['data']
    if data['fold_sha256'] != fold_sha256 or data['label_columns'] != list(LABEL_COLUMNS):
        raise ValueError(f'{name} used another fold file or label order.')
    if not np.array_equal(member['folds'], folds):
        raise ValueError(f'{name} assigns images to other folds.')
in_fold0 = folds == VALIDATION_FOLD
validation_ids = [image_id for image_id, is_fold0 in zip(train_ids, in_fold0) if is_fold0]
validation_targets = train_targets[in_fold0]

validation_probabilities, test_probabilities, member_sources = {}, {}, {}
for name, (experiment, run_id) in FOLD0_MEMBERS.items():
    run_dir = ROOT / 'experiments' / experiment / 'results' / run_id
    record = json.loads((run_dir / f'{run_id}.json').read_text())
    if record['fold'] != VALIDATION_FOLD or record['data']['fold_sha256'] != fold_sha256 or record['data']['label_columns'] != list(LABEL_COLUMNS):
        raise ValueError(f'{run_id} is not a fold-0 run on the same folds and labels.')
    validation_probabilities[name] = read_probabilities(run_dir / f'{run_id}_val.csv', validation_ids)
    test_csv = ROOT / 'experiments' / experiment / 'results' / f'{run_id}_test' / f'{run_id}_test.csv'
    test_probabilities[name] = read_probabilities(test_csv, test_ids)
    member_sources[name] = {'kind': 'fold-0 model', 'run_id': run_id, 'test_csv_sha256': sha256(test_csv)}
for name, member in cv_members.items():
    validation_probabilities[name] = member['oof'][in_fold0]
    test_probabilities[name] = member['test']
    member_sources[name] = {'kind': 'five-fold mean', 'folder': f'experiments/cv/{CV_MEMBERS[name]}', 'cv_record_sha256': sha256(ROOT / 'experiments' / 'cv' / CV_MEMBERS[name] / 'cv.json'),
                            'fold_sources': member['record']['fold_sources']}
MEMBERS = list(validation_probabilities)
print('Members:', MEMBERS, f'| fold-0 images: {len(validation_ids)} | test images: {len(test_ids)}')
for name in CV_MEMBERS:
    print(name, 'fold sources:', cv_members[name]['record']['fold_sources'])

Members: ['exp1', 'exp07', 'cv_exp3', 'cv_exp5'] | fold-0 images: 2452 | test images: 18673
cv_exp3 fold sources: {'0': 'imported', '1': 'trained', '2': 'trained', '3': 'trained', '4': 'trained'}
cv_exp5 fold sources: {'0': 'imported', '1': 'trained', '2': 'trained', '3': 'trained', '4': 'trained'}


## Sanity check on fold 0

Fold 0 is the only fold with predictions from all four members. The fold-0 model of each five-fold member is the old fold-0 run, imported unchanged (see the fold sources above). So the four fold-0 predictions are the same as in notebook 18, and the mean must again score 0.99832. This checks that the files load and align. It cannot rank the ensembles: notebook 18 showed that fold 0 points the wrong way for this question.

In [3]:
candidates = {**validation_probabilities, 'mean': mean_probabilities([validation_probabilities[name] for name in MEMBERS])}
print(f"{'model':<8} {'F1@0.25':>8} " + ' '.join(f'{label:>8}' for label in LABEL_COLUMNS))
for name, probabilities in candidates.items():
    metrics = score_probabilities(validation_targets, probabilities, THRESHOLD)
    print(f"{name:<8} {metrics['macro_f1']:>8.5f} " + ' '.join(f"{metrics['per_label_f1'][label]:>8.4f}" for label in LABEL_COLUMNS))

model     F1@0.25      Cat      Dog    Moris    Motya  Biatrix
exp1      0.97518   0.9966   0.9970   1.0000   0.9412   0.9412
exp07     0.99733   0.9941   0.9925   1.0000   1.0000   1.0000
cv_exp3   0.99831   0.9971   0.9945   1.0000   1.0000   1.0000
cv_exp5   0.99851   0.9970   0.9955   1.0000   1.0000   1.0000
mean      0.99832   0.9956   0.9960   1.0000   1.0000   1.0000


## What changes on the test set

Here the members differ from notebook 18: Exp3 and Exp5 now predict the test set with the mean of five fold models. We count the test cells (image, label) where the new ensemble differs from the old four-model mean and from our best submission. One named-pet cell is worth about 0.0007 public macro F1, and a Cat or Dog cell about 0.00001 (notebook 19), so the named-pet columns matter most. The second table shows how many members vote for each named-pet positive.

In [4]:
named_pets = [LABEL_COLUMNS.index(label) for label in ('Moris', 'Motya', 'Biatrix')]
ensemble_test = mean_probabilities([test_probabilities[name] for name in MEMBERS])
decided = ensemble_test >= THRESHOLD

compared = {'this notebook': decided}
for title, path in EARLIER.items():
    rows = {row['image_id']: row for row in read_csv(ROOT / path, ('image_id', *LABEL_COLUMNS))}
    compared[title] = np.asarray([[int(rows[image_id][label]) for label in LABEL_COLUMNS] for image_id in test_ids], dtype=bool)

print(f"{'test positives':<38}" + ''.join(f'{label:>8}' for label in LABEL_COLUMNS))
for title, labels in compared.items():
    print(f'{title:<38}' + ''.join(f'{int(labels[:, index].sum()):>8}' for index in range(len(LABEL_COLUMNS))))
print('\nCells where this notebook differs (added/removed positives):')
for title in EARLIER:
    added, removed = decided & ~compared[title], ~decided & compared[title]
    print(f'  vs {title:<37} ' + ''.join(f'{label} +{int(added[:, i].sum())}/-{int(removed[:, i].sum())}  ' for i, label in enumerate(LABEL_COLUMNS)))

votes = np.sum([test_probabilities[name][:, named_pets] >= THRESHOLD for name in MEMBERS], axis=0)
print('\nNamed-pet positives by the number of members that vote for them:')
for position, label_index in enumerate(named_pets):
    positive = decided[:, label_index]
    print(f'  {LABEL_COLUMNS[label_index]:<8} {int(positive.sum()):>4}: ' + ', '.join(f'{count} members {int((positive & (votes[:, position] == count)).sum())}' for count in range(1, len(MEMBERS) + 1)))

test positives                             Cat     Dog   Moris   Motya Biatrix
this notebook                             7817    7711     173     139     183
old 4-mean (public 0.97816)               7825    7700     165     138     184
best, skill weights (public 0.98305)      7826    7701     170     135     185

Cells where this notebook differs (added/removed positives):
  vs old 4-mean (public 0.97816)           Cat +11/-19  Dog +24/-13  Moris +9/-1  Motya +3/-2  Biatrix +1/-2  
  vs best, skill weights (public 0.98305)  Cat +9/-18  Dog +22/-12  Moris +5/-2  Motya +4/-0  Biatrix +0/-2  

Named-pet positives by the number of members that vote for them:
  Moris     173: 1 members 2, 2 members 11, 3 members 22, 4 members 138
  Motya     139: 1 members 1, 2 members 7, 3 members 14, 4 members 117
  Biatrix   183: 1 members 0, 2 members 5, 3 members 11, 4 members 167


## Write the submission

The CSV goes to `submissions/` in sample-submission order. A record with the members and their file hashes goes to `experiments/cv/ensemble/<run-id>/`, next to the five-fold ensembles of notebook `cv/06`. Nothing is overwritten. Upload by hand and write the public score into the record and the table below.

In [5]:
if BUILD_SUBMISSION:
    submission_path = ROOT / 'submissions' / f'{RUN_ID}.csv'
    record_path = ROOT / 'experiments' / 'cv' / 'ensemble' / RUN_ID / 'record.json'
    for path in (submission_path, record_path):
        if path.exists():
            raise FileExistsError(f'Refusing to overwrite {path}')
    git_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=ROOT, capture_output=True, text=True).stdout.strip() or None
    write_submission(submission_path, form_submission(DATA_DIR / 'sample_submission.csv', test_ids, ensemble_test, THRESHOLD))
    write_run_record(record_path, {
        'run_id': RUN_ID, 'status': 'completed', 'method': 'equal-weight mean of member test probabilities, then fixed threshold',
        'threshold': THRESHOLD, 'members': member_sources,
        'data': {'fold_file': 'folds.csv', 'fold_sha256': fold_sha256, 'label_columns': list(LABEL_COLUMNS)},
        'submission_csv': str(submission_path.relative_to(ROOT)), 'submission_sha256': sha256(submission_path),
        'git_commit': git_commit, 'public_leaderboard_score': None,
    })
    print('CSV ready for manual upload:', submission_path.relative_to(ROOT))
    print('Record:', record_path.relative_to(ROOT))
else:
    print('No submission CSV built.')

CSV ready for manual upload: submissions/cv_ensemble_exp1_cvexp3_cvexp5_exp7_mean_025.csv
Record: experiments/cv/ensemble/cv_ensemble_exp1_cvexp3_cvexp5_exp7_mean_025/record.json


## Results and conclusions

_Fill in after the upload._

| Submission | Exp3, Exp5 | Public LB |
|---|---|---:|
| Mean of Exp1, Exp3, Exp5, Exp7 (notebook 18) | fold-0 models | 0.97816 |
| Mean of Exp1, Exp3, Exp5, Exp7 (this notebook) | five-fold means | |
| Best so far: `skill` weights (notebook 19) | fold-0 models | 0.98305 |